In [ ]:
import matplotlib.pyplot as plt
import cmath
import math
import meep as mp
from IPython.display import Video

: 

In [ ]:
# 3D simulation, planewave in y direction, E field in x direction, 15-degree tilt (from vertical)
theta_src = math.radians(15)  # Elevation angle (from vertical)
phi_src = math.radians(15)    # Azimuthal angle
um_scale = 1.0                # Microns per micron
frequency_G = (1/(0.30 * um_scale) + 1/(0.80 * um_scale))/2
o_grid = 1/50 # 20nm 
sp_size = round(o_grid * 28, 2) # subpixel size 560 nm

wvl = 1                       # Wavelength
fcen = frequency_G                # Frequency center

resolution = 20 / wvl         # Spatial resolution

if True: ##### Set main parameters ############################################################################################################################                                                                                                                                                                                                       
    fl_thickness = round(o_grid * 25, 2) # focal layer thickness 500 nm              

    
    ml_thickness = round(o_grid * 20, 2) # multi layer thickness 200 nm 
    el_thickness = round(o_grid * 2, 2) # etch layer thickness 40 nm

    sp_size = round(o_grid * 28, 2) # subpixel size 560 nm
    sd_size = round(sp_size, 2) # real SiPD size

if True: #####  Set main components and monitors  #############################################################################################################
    Lpml = round(o_grid * 10, 2) # PML thickness 200 nm
    pml_layers = [mp.Absorber(thickness = Lpml, direction = mp.Z)]

    pml_2_src = round(o_grid * 5, 2) # PML to source 100 nm
    src_2_geo = round(o_grid * 5, 2) # Source to geometry 100 nm
    mon_2_pml = round(o_grid * 10, 2) # Monitor to PML 200 nm

    # Design region size
    design_region_width_x = round(sp_size * 2 , 2) # Design region x 1120 nm
    design_region_width_y = round(sp_size * 2 , 2) # Design region y 1120 nm
    design_region_height = round(ml_thickness * 5  + el_thickness * (5-1), 2) # Design region z 1160 nm   

    # Overall cell size
    Sx = design_region_width_x
    Sy = design_region_width_y
    Sz = round(Lpml +  pml_2_src + src_2_geo + design_region_height + fl_thickness +  mon_2_pml + Lpml, 2)
    cell_size = mp.Vector3(Sx, Sy, Sz)

# PML layers along Z
pml_layers = [mp.PML(thickness=Lpml, direction=mp.Z)]

# Planewave amplitude function
def pw_amp(k, x0):
    def _pw_amp(x):
        return cmath.exp(1j * 2 * math.pi * k.dot(x + x0))
    return _pw_amp

# Wavevector for Z propagation
k = mp.Vector3(
    math.sin(theta_src) * math.cos(phi_src),  # kx component
    math.sin(theta_src) * math.sin(phi_src),  # ky component
    math.cos(theta_src)                      # kz component
).scale(fcen)

# If the source is perfectly vertical (theta_src == 0), k points only in Z
if theta_src == 0:
    k = mp.Vector3(0, 0, 0)

src_pt = [0, 0, round(Sz / 2 - Lpml - pml_2_src, 2) ]

sources = [mp.Source(mp.GaussianSource(fcen, fwidth=1 /(0.30 * um_scale) - 1/(0.80 * um_scale)),
                     component=mp.Ex,
                     center=src_pt,
                     size=mp.Vector3(x=Sx, y=Sy),
                     amp_func=pw_amp(k, src_pt))]

sim = mp.Simulation(cell_size=cell_size,
                    sources=sources,
                    k_point=k,
                    boundary_layers=pml_layers,
                    resolution=resolution)

t = 10  # run time
f = plt.figure(dpi=150)
volume = mp.Block(size=mp.Vector3(Sx,0,Sz), center=mp.Vector3(y=int(Sy/2)))
Animate = mp.Animate2D(sim, output_plane=volume, fields=mp.Ex, f=f, realtime=False, normalize=True)
sim.run(mp.at_every(0.1, Animate), until=t)

filename = "test.mp4"
Animate.to_mp4(10,filename)
Video(filename)

/root/miniconda3/envs/mp/lib/python3.8/site-packages/meep/visualization.py:1446: UserWarning: Warning: The 'sim' argument in Animate2D is deprecated and has no effect. It will be removed in a future release.
  warnings.warn(


-----------
Initializing structure...
time for choose_chunkdivision = 1.81198e-05 s
Working in 3D dimensions.
Computational cell is 1.1 x 1.1 x 3.45 with resolution 20
time for set_epsilon = 0.03269 s
-----------
